In [1]:
import os
import pandas as pd
import swifter 
import numpy as np
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split


nltk_path = os.path.join(os.path.dirname(nltk.__file__), 'nltk_data')
nltk.data.path.append(nltk_path)
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

force_prep = False # set to True if we want to preprocess data by force

if os.path.exists('prepped_articles.csv') and not force_prep:
    df = df = pd.read_csv('prepped_articles.csv')
    X_train, X_test, y_train, y_test = train_test_split(df['text'], df['label'], test_size=0.2, random_state=42)
else:
    df = pd.read_csv('articles.csv')

    X_train, X_test, y_train, y_test = train_test_split(df['text'], df['label'], test_size=0.2, random_state=42)

    stop_words = set(stopwords.words('english'))
    def preprocess(txt):
        txt = txt.lower()
        txt = ''.join([c for c in txt if c not in string.punctuation])
        toks = word_tokenize(txt)
        toks = [word for word in toks if word not in stop_words]
        return ' '.join(toks)

    X_train = X_train.swifter.apply(preprocess)
    X_test = X_test.swifter.apply(preprocess)

    df_prepped = pd.DataFrame({'text': pd.concat([X_train, X_test]), 'label': pd.concat([y_train, y_test])})
    df_prepped.to_csv('prepped_articles.csv', index=False)
X_train
X_test

[nltk_data] Downloading package stopwords to /home/alex/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/alex/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/alex/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


6480     justice departments decision water recommendat...
431      white house press secretary sarah huckabee san...
12412    new york ap — embarrassing aboutface new york ...
836      president trump issued two executive orders la...
13032    west palm beach florida cnnpresident donald tr...
                               ...                        
4844     congress wednesdays protest dramatic escalatio...
7677     people list best news stories 2020 would extre...
12276    partisan enmity incendiary rhetoric polarizati...
3559     shanghai—family friends rushed hospitals thurs...
7040     like father former rep ron paul sen rand paul ...
Name: text, Length: 3473, dtype: object

In [2]:
from keras._tf_keras.keras.preprocessing.text import Tokenizer
from keras._tf_keras.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer()
tokenizer.fit_on_texts(X_train)
X_train = tokenizer.texts_to_sequences(X_train)
X_test = tokenizer.texts_to_sequences(X_test)

max_length = 100
X_train = pad_sequences(X_train, maxlen=max_length, padding='post')
X_test = pad_sequences(X_test, maxlen=max_length, padding='post')

2025-09-24 02:43:29.476484: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-24 02:43:29.505433: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758653009.524149   29848 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758653009.529871   29848 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1758653009.551259   29848 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [3]:
# Load GloVe
embeddings_index = {}
with open('glove/glove.6B.100d.txt', encoding='utf8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs

print(f'Found {len(embeddings_index)} word vectors.')


Found 400000 word vectors.


In [4]:
# Prepare embedding matrix
embedding_dim = 100
word_index = tokenizer.word_index
num_words = len(word_index) + 1
embedding_matrix = np.zeros((num_words, embedding_dim))

for word, i in word_index.items():
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector

In [5]:
# Search for nearest words using cosine similarity
def cosine_similarity(vec1, vec2):
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

test_words = ['doomscrolling', 'brat', 'cryptowallet']

for word in test_words:
    if word in embeddings_index:
        vec = embeddings_index[word]
        similarities = {}
        for w, v in embeddings_index.items():
            if w != word:
                similarities[w] = cosine_similarity(vec, v)
        
        top3 = sorted(similarities.items(), key=lambda x: x[1], reverse=True)[:3]
        print(f'"{word}" -> Closest words: {[w for w, sim in top3]}')
    else:
        print(f'"{word}" not found in vocabulary.')


"doomscrolling" not found in vocabulary.
"brat" -> Closest words: ['brats', 'nerd', 'slacker']
"cryptowallet" not found in vocabulary.


In [6]:
# SOP 1: GloVe doesn't embed OOV words

test_words = ['othering', 'hate-fueled']

for w in test_words:
    if w in embeddings_index:
        print(f"'{w}' is IN vocabulary")
    else:
        print(f"'{w}' is not in GloVe vocabulary")

# "othering": https://www.scientificamerican.com/article/why-hatred-and-othering-of-political-foes-has-spiked-to-extreme-levels/
# "hate-fueled": https://people.com/gretchen-whitmer-statement-trump-shooting-political-division-8677954 


'othering' is not in GloVe vocabulary
'hate-fueled' is not in GloVe vocabulary


In [7]:
# SOP 2: GloVe has no awareness of word morphology
from scipy.spatial.distance import cosine

word_pairs = [('democrat', 'democratization'), ('govern','government'), ('legislate','legislation'), ('voted', 'voter'), ('elect','election'), ('woke', 'liberal')]

for w1, w2 in word_pairs:
    if w1 in embeddings_index and w2 in embeddings_index:
        sim = 1 - cosine(embeddings_index[w1], embeddings_index[w2])
        print(f"Similarity({w1}, {w2}) = {sim:.3f}")
    else:
        print(f"{w1} or {w2} not in vocab")

Similarity(democrat, democratization) = 0.006
Similarity(govern, government) = 0.441
Similarity(legislate, legislation) = 0.383
Similarity(voted, voter) = 0.438
Similarity(elect, election) = 0.662
Similarity(woke, liberal) = -0.054


In [8]:
# SOP 3: GloVe struggles to capture Semantic Relationships and Similarities between words
antonym_pairs = [("democrat", "republican"), ("increase", "decrease"), ("good", "bad"), ('violence','peace'), ('division', 'unity')]

for w1, w2 in antonym_pairs:
    if w1 in embeddings_index and w2 in embeddings_index:
        sim = 1 - cosine(embeddings_index[w1], embeddings_index[w2])
        print(f"Similarity({w1}, {w2}) = {sim:.3f}")


Similarity(democrat, republican) = 0.879
Similarity(increase, decrease) = 0.881
Similarity(good, bad) = 0.770
Similarity(violence, peace) = 0.533
Similarity(division, unity) = 0.249


In [9]:
polusa_20191 = pd.read_csv("polusa/2019_1.csv")
polusa_20191.head()


,id,date_publish,outlet,headline,lead,body,authors,domain,url,political_leaning
0,53010215,2019-01-01 00:00:00,NPR,What We Know About The American Russia Has Det...,The U.S. and Russia are beginning the new year...,What We Know About The American Russia Has Det...,Greg Myre,www.npr.org,https://www.npr.org/2019/01/01/681442511/what-...,LEFT
1,59549287,2019-01-01 00:00:00,Chicago Tribune,NASA's New Horizons spacecraft just visited th...,NaN,The nerdiest New Year's party in the solar sys...,Sarah Kaplan,www.chicagotribune.com,https://www.chicagotribune.com/news/nationworl...,UNDEFINED
2,59633617,2019-01-01 00:00:00,Chicago Tribune,"Steven Lattimore, Chicago journalist and teach...",NaN,Chicago journalist Steven Lattimore worked in ...,Graydon Megan,www.chicagotribune.com,https://www.chicagotribune.com/news/local/poli...,UNDEFINED
3,52963105,2019-01-01 00:00:00,NPR,"Kim Jong Un Wants New Summit With Trump, But A...",In the North Korean leader's New Year's addres...,"Kim Jong Un Wants New Summit With Trump, But A...",Scott Neuman,www.npr.org,https://www.npr.org/2019/01/01/681398273/kim-j...,LEFT
4,18321756,2019-01-01 00:00:00,BBC,What home comforts keep this UN peacekeeper go...,UN peacekeeper Major Michelle Kayanda from Zam...,Video\nUN peacekeeper Major Michelle Kayanda f...,NaN,www.bbc.com,http://www.bbc.com/news/av/world-africa-423689...,UNDEFINED


In [10]:
find_article = polusa_20191
matches = find_article[find_article['body'].str.contains(r'\bhate-fueled\b', case=False, na=False)]

print(matches)             
print(len(matches))         

               id         date_publish     outlet  \
21354     4107012  2019-01-24 10:00:02  USA Today   
67186   131723725  2019-03-18 20:18:57      Slate   
69304     4338970  2019-03-20 18:58:25  Breitbart   
69493   113893701  2019-03-20 22:39:11   CBS News   
101047   55279713  2019-04-25 00:24:00   NBC News   
104319    4143872  2019-04-29 10:44:08  USA Today   
104462    4180651  2019-04-29 14:23:12  USA Today   
107497    4157851  2019-05-02 04:01:09  USA Today   
109212    4160063  2019-05-03 15:38:26  USA Today   
118487  113846627  2019-05-13 22:20:03   CBS News   
118530    3881347  2019-05-13 23:24:04   HuffPost   
118843    4683328  2019-05-14 04:55:04        CNN   
129477    4266701  2019-05-24 08:00:07  USA Today   
138706    4128000  2019-06-04 10:00:08  USA Today   
145540    4255599  2019-06-11 12:07:18  USA Today   
152474  113893673  2019-06-18 20:15:12   CBS News   
153703  115972453  2019-06-19 22:24:00   ABC News   

                                             